In [265]:
import pandas as pd

# 1. load
receivals = pd.read_csv("data/kernel/receivals.csv")
purchase = pd.read_csv("data/kernel/purchase_orders.csv")
pred_map  = pd.read_csv("data/prediction_mapping.csv")

# 2. parse dates
receivals["date_arrival"] = pd.to_datetime(receivals["date_arrival"], utc=True).dt.tz_convert(None)
purchase["delivery_date"] = pd.to_datetime(purchase["delivery_date"], utc=True).dt.tz_convert(None)

# 3. make the ID columns nullable ints (so NaN is allowed)
receivals["purchase_order_id"] = receivals["purchase_order_id"].astype("Int64")
receivals["purchase_order_item_no"] = receivals["purchase_order_item_no"].astype("Int64")

purchase["purchase_order_id"] = purchase["purchase_order_id"].astype("Int64")
purchase["purchase_order_item_no"] = purchase["purchase_order_item_no"].astype("Int64")

# 4. build the subset that actually has PO info, and only there cast to plain int
rec_po = receivals.dropna(subset=["purchase_order_id", "purchase_order_item_no"]).copy()
rec_po["purchase_order_id"] = rec_po["purchase_order_id"].astype(int)
rec_po["purchase_order_item_no"] = rec_po["purchase_order_item_no"].astype(int)

# cast purchase ids too
purchase["purchase_order_id"] = purchase["purchase_order_id"].astype(int)
purchase["purchase_order_item_no"] = purchase["purchase_order_item_no"].astype(int)


In [266]:
# map PO-line -> rm_id from receivals
po_rm_map = (
    rec_po[["purchase_order_id", "purchase_order_item_no", "rm_id"]]
    .drop_duplicates()
)

# attach rm_id to purchase orders
purchase_rm = purchase.merge(
    po_rm_map,
    on=["purchase_order_id", "purchase_order_item_no"],
    how="left",
)

# re-read receivals? (you don't need to, but keeping your style)
# we'll just reuse the one we already parsed
receivals["date"] = receivals["date_arrival"].dt.normalize()

daily_recv = (
    receivals
    .groupby(["rm_id", "date"])["net_weight"]
    .sum()
    .reset_index()
    .rename(columns={"net_weight": "daily_net_weight"})
)


In [267]:
# calendar over all rm_id x date
min_date = daily_recv["date"].min()
max_date = daily_recv["date"].max()
all_days = pd.date_range(min_date, max_date, freq="D")
rm_ids = daily_recv["rm_id"].unique()

frames = []
for rm in rm_ids:
    df = pd.DataFrame({"date": all_days})
    df["rm_id"] = rm
    frames.append(df)

full_daily = pd.concat(frames, ignore_index=True)

# join actual deliveries
full_daily = full_daily.merge(daily_recv, on=["rm_id", "date"], how="left").sort_values(["rm_id", "date"])
full_daily["daily_net_weight"] = full_daily["daily_net_weight"].fillna(0.0)

In [268]:
def add_lags(df, group="rm_id"):
    df = df.sort_values([group, "date"]).copy()
    df["daily_net_weight"] = df["daily_net_weight"].fillna(0.0)

    for lag in [1, 2, 7, 14, 30]:
        df[f"lag_{lag}"] = df.groupby(group)["daily_net_weight"].shift(lag)

    df["roll_7"] = (
        df.groupby(group)["daily_net_weight"]
          .rolling(7, min_periods=1)
          .sum()
          .reset_index(level=0, drop=True)
    )
    df["roll_30"] = (
        df.groupby(group)["daily_net_weight"]
          .rolling(30, min_periods=1)
          .sum()
          .reset_index(level=0, drop=True)
    )
    return df

full_daily = add_lags(full_daily)

In [269]:
pred_map["date"] = pd.to_datetime(pred_map["forecast_end_date"])
future = pred_map[["rm_id", "date"]].drop_duplicates()

full_plus = pd.concat([full_daily, future], ignore_index=True)
full_plus = full_plus.drop_duplicates(["rm_id", "date"]).sort_values(["rm_id", "date"])
full_plus["daily_net_weight"] = full_plus["daily_net_weight"].fillna(0.0)

# re-run lags on extended
full_plus = add_lags(full_plus)

In [270]:
def make_po_features_fast(base_df, po_df, horizon_days=(30, 60, 150)):
    # find qty col
    qty_col = None
    for c in ["quantity", "order_qty", "ordered_qty"]:
        if c in po_df.columns:
            qty_col = c
            break
    if qty_col is None:
        out = base_df[["rm_id", "date"]].copy()
        for h in horizon_days:
            out[f"po_qty_{h}"] = 0.0
        return out

    po = po_df[["rm_id", "delivery_date", qty_col]].copy()
    po["delivery_date"] = pd.to_datetime(po["delivery_date"], utc=True).dt.tz_convert(None)

    po_day = (
        po.groupby(["rm_id", "delivery_date"])[qty_col]
          .sum()
          .reset_index()
          .rename(columns={"delivery_date": "date", qty_col: "po_qty"})
    )

    cal = base_df[["rm_id", "date"]].merge(po_day, on=["rm_id", "date"], how="left")
    cal["po_qty"] = cal["po_qty"].fillna(0.0)
    cal = cal.sort_values(["rm_id", "date"])
    out = cal[["rm_id", "date"]].copy()

    for h in horizon_days:
        tmp = (
            cal.sort_values(["rm_id", "date"], ascending=[True, False])
               .groupby("rm_id")["po_qty"]
               .rolling(h, min_periods=1)
               .sum()
               .reset_index(level=0, drop=True)
        )
        col = f"po_qty_{h}"
        cal[col] = tmp.values
        cal = cal.sort_values(["rm_id", "date"])
        out[col] = cal[col].values

    for h in horizon_days:
        col = f"po_qty_{h}"
        if col not in out.columns:
            out[col] = 0.0

    return out

po_feats = make_po_features_fast(full_plus, purchase_rm, horizon_days=(30, 60, 150))
full_plus = full_plus.merge(po_feats, on=["rm_id", "date"], how="left")

for col in ["po_qty_30", "po_qty_60", "po_qty_150"]:
    full_plus[col] = full_plus[col].fillna(0.0)

In [271]:
# ---------- date features ----------
full_plus["dayofyear"] = full_plus["date"].dt.dayofyear
full_plus["month"] = full_plus["date"].dt.month
full_plus["dow"] = full_plus["date"].dt.dayofweek

In [272]:
# now make the training year slice
train_2024 = full_plus[full_plus["date"].dt.year == 2024].copy()
train_2024 = train_2024.dropna(subset=["lag_30", "roll_30"])

# target
train_2024["cum_from_jan1"] = (
    train_2024.groupby("rm_id")["daily_net_weight"].cumsum()
)

# drop early rows with NaN lags
train_2024 = train_2024.dropna(subset=["lag_30", "roll_30"])

target_col = "cum_from_jan1"
drop_cols = [target_col, "date"]
feature_cols = [c for c in train_2024.columns if c not in drop_cols]

# time split
cut_date = train_2024["date"].quantile(0.8)
train_mask = train_2024["date"] <= cut_date
valid_mask = ~train_mask

X_train = train_2024.loc[train_mask, feature_cols]
y_train = train_2024.loc[train_mask, target_col]
X_valid = train_2024.loc[valid_mask, feature_cols]
y_valid = train_2024.loc[valid_mask, target_col]

# make sure only numeric
X_train = X_train.select_dtypes(include=["number"])
X_valid = X_valid.select_dtypes(include=["number"])

In [273]:
import lightgbm as lgb

params = {
    "objective": "quantile",
    "alpha": 0.2,
    "metric": "quantile",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.7,
    "bagging_freq": 1,
    "verbose": -1,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
}

dtrain = lgb.Dataset(X_train, label=y_train)
dvalid = lgb.Dataset(X_valid, label=y_valid)


model = lgb.train(
    params,
    dtrain,
    num_boost_round=10000,
    valid_sets=[dtrain, dvalid],
    valid_names=["train", "valid"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=150),
        lgb.log_evaluation(period=50),
    ],
)

# use this when predicting
best_iter = model.best_iteration

Training until validation scores don't improve for 150 rounds
[50]	train's quantile: 21295.5	valid's quantile: 56535.7
[100]	train's quantile: 14221.1	valid's quantile: 43995.8
[150]	train's quantile: 11047.5	valid's quantile: 37338.3
[200]	train's quantile: 9871.87	valid's quantile: 35488.2
[250]	train's quantile: 9046.04	valid's quantile: 33977.5
[300]	train's quantile: 8553.5	valid's quantile: 33369.6
[350]	train's quantile: 7978.54	valid's quantile: 32106.1
[400]	train's quantile: 7458.75	valid's quantile: 31031.3
[450]	train's quantile: 7094.34	valid's quantile: 30372.4
[500]	train's quantile: 6845.13	valid's quantile: 30052
[550]	train's quantile: 6655.71	valid's quantile: 29810.3
[600]	train's quantile: 6443.52	valid's quantile: 29460.7
[650]	train's quantile: 6272.2	valid's quantile: 29073.1
[700]	train's quantile: 6062.33	valid's quantile: 28673.8
[750]	train's quantile: 5858.37	valid's quantile: 28176
[800]	train's quantile: 5740.62	valid's quantile: 28034
[850]	train's quant

In [274]:
# 1) you already have this
to_pred = pred_map.merge(full_plus, on=["rm_id", "date"], how="left")

model_feats = model.feature_name()
for f in model_feats:
    if f not in to_pred.columns:
        to_pred[f] = 0.0

X_test = to_pred[model_feats].fillna(0.0).astype(float)
y_pred = model.predict(X_test, num_iteration=model.best_iteration)
y_pred = y_pred.clip(0.0)

sample_sub = pd.read_csv("data/sample_submission.csv")
sub = sample_sub[["ID"]].merge(
    pred_map[["ID"]].assign(predicted_weight=y_pred),
    on="ID",
    how="left"
)
sub["predicted_weight"] = sub["predicted_weight"].fillna(0.0)
sub.to_csv("submission_lightgbm_alpha05.csv", index=False)

In [275]:

# base_params = {
#     "objective": "quantile",
#     "alpha": 0.2,
#     "metric": "quantile",
#     "learning_rate": 0.03,
#     "bagging_freq": 1,
#     "verbose": -1,
#     "feature_pre_filter": False,
# }

# grids = [
#     # current-ish
#     {"num_leaves": 63, "min_data_in_leaf": 100, "feature_fraction": 0.7, "bagging_fraction": 0.7, "lambda_l2": 0.0},
#     # smaller leaves
#     {"num_leaves": 31, "min_data_in_leaf": 50, "feature_fraction": 0.8, "bagging_fraction": 0.7, "lambda_l2": 0.0},
#     # more conservative leaf size
#     {"num_leaves": 63, "min_data_in_leaf": 200, "feature_fraction": 0.7, "bagging_fraction": 0.7, "lambda_l2": 0.0},
#     # add L2
#     {"num_leaves": 63, "min_data_in_leaf": 100, "feature_fraction": 0.8, "bagging_fraction": 0.6, "lambda_l2": 1.0},
#     # slower lr
#     {"num_leaves": 63, "min_data_in_leaf": 100, "feature_fraction": 0.7, "bagging_fraction": 0.7, "lambda_l2": 1.0, "learning_rate": 0.03},
# ]

# dtrain = lgb.Dataset(X_train, label=y_train)
# dvalid = lgb.Dataset(X_valid, label=y_valid)

# best_score = float("inf")
# best_params = None
# best_iter = None

# for i, g in enumerate(grids, 1):
#     params = base_params.copy()
#     params.update(g)
#     print(f"\n=== Try {i}/{len(grids)}: {g} ===")

#     model = lgb.train(
#         params,
#         dtrain,
#         num_boost_round=8000,
#         valid_sets=[dtrain, dvalid],
#         valid_names=["train", "valid"],
#         callbacks=[
#             lgb.early_stopping(stopping_rounds=100),
#             lgb.log_evaluation(period=200),
#         ],
#     )

#     val_score = model.best_score["valid"]["quantile"]
#     print("val quantile:", val_score)

#     if val_score < best_score:
#         best_score = val_score
#         best_params = params
#         best_iter = model.best_iteration

# print("\nBEST:", best_score)
# print("PARAMS:", best_params)
# print("ITER:", best_iter)